# jjh

---

**Project**
- CCRM

**Module**
- notebooks

**Author**
- Hyeok

**Created**
- 2026-03-07

**Purpose**
- TODO: EasyEnsemble
---


In [1]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os
import numpy as np
import pandas as pd
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score, accuracy_score
)

In [3]:
# 1. 환경 설정 및 데이터 로드
sys.path.append(os.path.abspath("../../"))
from common.fetch_creditcard import fetch_creditcard

df = fetch_creditcard()
X = df.drop(columns=["creditcard_churn_id", "churn"])
y = df["churn"]

In [4]:
# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
# [📊 요약 출력 - 팀 공통 규격]
print(f"\n{'Train / Test Split Summary':^50}")
print("="*50)
print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print("-"*50)
print(f"y_train shape : {y_train.shape}")
print(f"y_test  shape : {y_test.shape}")
print("-"*50)
print("Target Distribution (Train)")
print(y_train.value_counts().to_string())
print("-"*50)
print("Target Distribution (Test)")
print(y_test.value_counts().to_string())
print("-"*50)
print("Train Ratio")
print(y_train.value_counts(normalize=True).to_string())
print("-"*50)
print("Test Ratio")
print(y_test.value_counts(normalize=True).to_string())
print("="*50)


            Train / Test Split Summary            
X_train shape : (8181, 19)
X_test  shape : (2046, 19)
--------------------------------------------------
y_train shape : (8181,)
y_test  shape : (2046,)
--------------------------------------------------
Target Distribution (Train)
churn
0    6835
1    1346
--------------------------------------------------
Target Distribution (Test)
churn
0    1709
1     337
--------------------------------------------------
Train Ratio
churn
0    0.835472
1    0.164528
--------------------------------------------------
Test Ratio
churn
0    0.835288
1    0.164712


In [6]:
# 3. 모델 생성 및 학습
# n_estimators를 5로 조정하여 지표의 균형을 맞춤
model = EasyEnsembleClassifier(
    n_estimators=10, 
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

,n_estimators,10
,estimator,None
,warm_start,False
,sampling_strategy,'auto'
,replacement,False
,n_jobs,-1
,random_state,42
,verbose,0


In [7]:
# 4. 예측 및 임계값 조정
y_prob = model.predict_proba(X_test)[:, 1]
threshold = 0.45
y_pred = (y_prob >= threshold).astype(int)

In [8]:
# 5. 결과 출력
print("\n### BankChurner 모델 결과 ###")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_prob):.4f}")
print("\n[Classification Report]")
print(classification_report(y_test, y_pred))
print("="*50)


### BankChurner 모델 결과 ###
Accuracy : 0.7698
ROC-AUC  : 0.9701
PR-AUC   : 0.8868

[Classification Report]
              precision    recall  f1-score   support

           0       0.99      0.73      0.84      1709
           1       0.42      0.98      0.58       337

    accuracy                           0.77      2046
   macro avg       0.70      0.85      0.71      2046
weighted avg       0.90      0.77      0.80      2046

